# GNN Binding Affinity - PDBbind v2020

Trains a 3-layer **GIN** (Graph Isomorphism Network) on the PDBbind v2020 refined set
to predict protein-ligand binding affinity (pKd) from 3D complex graphs.

Featurization is identical to `backend/services/gnn_affinity.py` so the exported
weights can be loaded directly for inference.

In [ ]:
import subprocess, sys

def pip_install(*packages):
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")

pip_install("torch-geometric", "rdkit", "biopython", "scipy", "httpx", "nest_asyncio")

import torch_geometric
from rdkit import Chem
from Bio.PDB import PDBParser
print(f"torch-geometric {torch_geometric.__version__}, RDKit {Chem.rdBase.rdkitVersion}")

## 1. Load PDBbind v2020 Refined Set

In [ ]:
import os
from pathlib import Path

# from google.colab import drive
# drive.mount('/content/drive')

PDBBIND_DIR = Path("/content/drive/MyDrive/Colab Notebooks/refined-set")
INDEX_DIR   = Path("/content/drive/MyDrive/Colab Notebooks/index/index")
INDEX_FILE  = INDEX_DIR / "INDEX_refined_data.2020"

# Fallback: index nested inside refined-set
if not INDEX_FILE.exists():
    alt = PDBBIND_DIR / "index" / "INDEX_refined_data.2020"
    if alt.exists():
        INDEX_FILE = alt
        INDEX_DIR = alt.parent

USE_MOCK = not INDEX_FILE.exists()

if USE_MOCK:
    print("PDBbind data not found, generating mock dataset for testing.\n")

    MOCK_DIR = Path("/content/mock_pdbbind")
    (MOCK_DIR / "index").mkdir(parents=True, exist_ok=True)

    import random
    random.seed(42)

    mock_entries = []
    for i in range(100):
        pdb_code = f"mock{i:04d}"
        pkd = round(random.uniform(2.0, 12.0), 2)
        mock_entries.append(f"{pdb_code}  2.00  2020  {pkd}  Kd={10**(-pkd)*1e6:.1f}uM  //  mock entry {i}")

    mock_index_path = MOCK_DIR / "index" / "INDEX_refined_data.2020"
    mock_index_path.write_text("\n".join(mock_entries) + "\n")

    from rdkit import Chem
    from rdkit.Chem import AllChem

    smiles_pool = [
        "CC(=O)Oc1ccccc1C(=O)O",
        "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
        "CC(=O)Nc1ccc(O)cc1",
        "c1ccc(cc1)C(=O)O",
        "OC(=O)c1cccnc1",
    ]
    aa_names = ["ALA", "GLY", "LEU", "VAL", "SER", "THR", "ASP", "GLU", "LYS", "ARG"]

    for entry_line in mock_entries:
        pdb_code = entry_line.split()[0]
        entry_dir = MOCK_DIR / pdb_code
        entry_dir.mkdir(exist_ok=True)

        smi = random.choice(smiles_pool)
        mol = Chem.MolFromSmiles(smi)
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=int(pdb_code[-4:]))
        AllChem.MMFFOptimizeMolecule(mol)
        (entry_dir / f"{pdb_code}_ligand.sdf").write_text(Chem.MolToMolBlock(mol))

        conf = mol.GetConformer()
        cx = sum(conf.GetAtomPosition(a).x for a in range(mol.GetNumAtoms())) / mol.GetNumAtoms()
        cy = sum(conf.GetAtomPosition(a).y for a in range(mol.GetNumAtoms())) / mol.GetNumAtoms()
        cz = sum(conf.GetAtomPosition(a).z for a in range(mol.GetNumAtoms())) / mol.GetNumAtoms()

        pdb_lines = []
        for j in range(30):
            x = cx + random.uniform(-8, 8)
            y = cy + random.uniform(-8, 8)
            z = cz + random.uniform(-8, 8)
            aa = random.choice(aa_names)
            elem = random.choice(["C", "N", "O", "C", "C"])
            atom_name = {"C": "CA", "N": "N", "O": "O"}[elem]
            pdb_lines.append(
                f"ATOM  {j+1:5d} {atom_name:<4s} {aa:3s} A{j+1:4d}    "
                f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           {elem:>2s}"
            )
        pdb_lines.append("END")
        (entry_dir / f"{pdb_code}_protein.pdb").write_text("\n".join(pdb_lines) + "\n")

    PDBBIND_DIR = MOCK_DIR
    INDEX_FILE = mock_index_path
    INDEX_DIR = mock_index_path.parent
    print(f"Created {len(mock_entries)} mock complexes in {MOCK_DIR}")

else:
    complex_dirs = [d for d in PDBBIND_DIR.iterdir() if d.is_dir() and d.name != "index"]
    print(f"PDBbind refined set: {PDBBIND_DIR}")
    print(f"Index file: {INDEX_FILE}")
    print(f"Complexes: {len(complex_dirs)}")

In [ ]:
def parse_pdbbind_index(index_path: Path) -> dict:
    """Parse INDEX_refined_data.2020 → {pdb_code: pkd}.

    Format: pdb_code  resolution  year  pkd  Kd/Ki=...  //  reference
    """
    entries = {}
    with open(index_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            tokens = line.split("//")[0].split()
            if len(tokens) < 4:
                continue
            try:
                entries[tokens[0]] = float(tokens[3])
            except ValueError:
                continue
    return entries

pkd_map = parse_pdbbind_index(INDEX_FILE)
print(f"Parsed {len(pkd_map)} entries")

if pkd_map:
    values = list(pkd_map.values())
    print(f"pKd range: [{min(values):.2f}, {max(values):.2f}], mean: {sum(values)/len(values):.2f}")

## 1b. Re-dock PDBbind with DiffDock (Domain Adaptation)

Training on crystal poses causes a domain mismatch: the model sees perfect poses during training
but noisy DiffDock-predicted poses at inference. Re-docking each PDBbind complex through the same
DiffDock NIM API used in the backend aligns training and inference distributions.

~5000 complexes at 35 RPM takes ~2.5 hours. All results are cached to Drive so this only runs once.

`USE_DIFFDOCK_POSES = False` skips this step and trains on crystal poses (original behavior).

In [ ]:
import asyncio
import httpx
import time
import json
from collections import deque
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm.auto import tqdm

# ─── Configuration ───────────────────────────────────────────────────────────

USE_DIFFDOCK_POSES = True  # Set False to train on crystal poses (original behavior)

NVIDIA_API_KEY = ""  # <-- Paste your NVIDIA NIM API key here

DIFFDOCK_URL = "https://health.api.nvidia.com/v1/biology/mit/diffdock"
ASSETS_URL = "https://api.nvcf.nvidia.com/v2/nvcf/assets"
TIMEOUT = 90.0
MAX_CONCURRENT = 2
RPM_LIMIT = 35

# Cache directory for docked poses (so you don't re-dock on reruns)
DOCKED_DIR = Path("/content/drive/MyDrive/Colab Notebooks/diffdock_poses")
if USE_DIFFDOCK_POSES:
    DOCKED_DIR.mkdir(parents=True, exist_ok=True)

# ─── Rate limiter (same as production) ───────────────────────────────────────

class RateLimiter:
    def __init__(self, max_requests, window_seconds=60.0):
        self._max = max_requests
        self._window = window_seconds
        self._timestamps = deque()
        self._lock = asyncio.Lock()

    async def acquire(self):
        async with self._lock:
            now = time.monotonic()
            while self._timestamps and self._timestamps[0] <= now - self._window:
                self._timestamps.popleft()
            if len(self._timestamps) >= self._max:
                sleep_until = self._timestamps[0] + self._window
                delay = sleep_until - now
                if delay > 0:
                    await asyncio.sleep(delay)
                self._timestamps.popleft()
            self._timestamps.append(time.monotonic())

# ─── DiffDock helpers ────────────────────────────────────────────────────────

def smiles_to_sdf(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:
        return None
    AllChem.MMFFOptimizeMolecule(mol)
    return Chem.MolToMolBlock(mol)


async def upload_asset(client, api_key, content, limiter):
    await limiter.acquire()
    resp = await client.post(
        ASSETS_URL,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"contentType": "text/plain", "description": "diffdock-file"},
    )
    resp.raise_for_status()
    data = resp.json()

    await limiter.acquire()
    resp = await client.put(
        data["uploadUrl"],
        content=content.encode(),
        headers={"x-amz-meta-nvcf-asset-description": "diffdock-file", "Content-Type": "text/plain"},
    )
    resp.raise_for_status()
    return data["assetId"]


async def dock_single(client, semaphore, limiter, api_key, protein_asset_id, sdf_text):
    """Dock a single ligand. Returns best-pose SDF string or None."""
    async with semaphore:
        try:
            await limiter.acquire()
            ligand_asset_id = await upload_asset(client, api_key, sdf_text, limiter)

            await limiter.acquire()
            resp = await client.post(
                DIFFDOCK_URL,
                json={
                    "ligand": ligand_asset_id,
                    "ligand_file_type": "sdf",
                    "protein": protein_asset_id,
                    "num_poses": 5,
                    "time_divisions": 20,
                    "steps": 18,
                    "save_trajectory": False,
                    "is_staged": True,
                },
                headers={
                    "Authorization": f"Bearer {api_key}",
                    "Content-Type": "application/json",
                    "NVCF-INPUT-ASSET-REFERENCES": f"{protein_asset_id},{ligand_asset_id}",
                },
            )

            if resp.status_code != 200:
                return None

            data = resp.json()
            if data.get("status") == "failed":
                return None

            positions = data.get("ligand_positions", [])
            confidences = data.get("position_confidence", [])

            valid = [(i, c) for i, c in enumerate(confidences)
                     if c is not None and i < len(positions) and positions[i]]
            if not valid:
                return None

            best_idx, _ = max(valid, key=lambda p: p[1])
            return positions[best_idx]

        except Exception as e:
            print(f"  Dock failed: {e}")
            return None


async def redock_batch(pdb_codes, pkd_map, pdbbind_dir, docked_dir, api_key):
    """Re-dock all PDBbind complexes with DiffDock. Caches results to disk."""
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    limiter = RateLimiter(RPM_LIMIT)

    # Figure out which ones still need docking
    todo = []
    for code in pdb_codes:
        cache_path = docked_dir / f"{code}_docked.sdf"
        fail_path = docked_dir / f"{code}_failed.txt"
        if cache_path.exists() or fail_path.exists():
            continue
        todo.append(code)

    already_done = len(pdb_codes) - len(todo)
    print(f"Already cached: {already_done}, remaining: {len(todo)}")

    if not todo:
        print("All complexes already docked!")
        return

    # Process in batches grouped by protein (1 protein upload per complex)
    async with httpx.AsyncClient(timeout=TIMEOUT) as client:
        for i, code in enumerate(tqdm(todo, desc="Re-docking with DiffDock")):
            cache_path = docked_dir / f"{code}_docked.sdf"
            fail_path = docked_dir / f"{code}_failed.txt"

            pdb_path = pdbbind_dir / code / f"{code}_protein.pdb"
            sdf_path = pdbbind_dir / code / f"{code}_ligand.sdf"

            if not pdb_path.exists() or not sdf_path.exists():
                fail_path.write_text("missing files")
                continue

            # Extract SMILES from crystal ligand
            crystal_sdf = sdf_path.read_text()
            mol = Chem.MolFromMolBlock(crystal_sdf, removeHs=True)
            if mol is None:
                fail_path.write_text("rdkit parse failed")
                continue

            smiles = Chem.MolToSmiles(mol)
            if not smiles or len(smiles) > 600:
                fail_path.write_text(f"bad smiles: len={len(smiles) if smiles else 0}")
                continue

            # Convert SMILES back to SDF (same as production pipeline)
            input_sdf = smiles_to_sdf(smiles)
            if input_sdf is None:
                fail_path.write_text("smiles_to_sdf failed")
                continue

            try:
                # Upload protein for this complex
                pdb_text = pdb_path.read_text()
                protein_asset = await upload_asset(client, api_key, pdb_text, limiter)

                # Dock
                docked_sdf = await dock_single(
                    client, semaphore, limiter, api_key, protein_asset, input_sdf
                )

                if docked_sdf:
                    cache_path.write_text(docked_sdf)
                else:
                    fail_path.write_text("docking returned no poses")

            except Exception as e:
                fail_path.write_text(str(e))
                print(f"  {code} failed: {e}")

            # Progress update every 100
            if (i + 1) % 100 == 0:
                done = already_done + i + 1
                print(f"  Progress: {done}/{len(pdb_codes)} ({done/len(pdb_codes)*100:.1f}%)")


# ─── Run re-docking ──────────────────────────────────────────────────────────

if USE_DIFFDOCK_POSES:
    if not NVIDIA_API_KEY:
        print("⚠️  Set NVIDIA_API_KEY above to run DiffDock re-docking!")
        print("   Falling back to crystal poses for now.")
        USE_DIFFDOCK_POSES = False
    else:
        pdb_codes_to_dock = sorted(pkd_map.keys())
        print(f"Re-docking {len(pdb_codes_to_dock)} PDBbind complexes with DiffDock...")
        print(f"Cache dir: {DOCKED_DIR}")

        # In Colab, use the existing event loop
        try:
            loop = asyncio.get_event_loop()
            if loop.is_running():
                import nest_asyncio
                nest_asyncio.apply()
        except RuntimeError:
            pass

        asyncio.run(redock_batch(pdb_codes_to_dock, pkd_map, PDBBIND_DIR, DOCKED_DIR, NVIDIA_API_KEY))

        # Report stats
        docked = len(list(DOCKED_DIR.glob("*_docked.sdf")))
        failed = len(list(DOCKED_DIR.glob("*_failed.txt")))
        print(f"\nDiffDock re-docking complete: {docked} docked, {failed} failed")
        print(f"Coverage: {docked}/{len(pdb_codes_to_dock)} ({docked/len(pdb_codes_to_dock)*100:.1f}%)")
else:
    print("Using crystal poses (USE_DIFFDOCK_POSES = False)")

## 2. Featurization

Each complex → PyG graph with **47-dim node features** and **24-dim edge features**.
Identical to `backend/services/gnn_affinity.py`.

In [ ]:
import numpy as np
import torch
from torch_geometric.data import Data
from rdkit import Chem
from Bio.PDB import PDBParser
from scipy.spatial.distance import cdist
import io, warnings

# ─── Constants ───────────────────────────────────────────────────────────────

ELEMENT_LIST = [6, 7, 8, 16, 15, 9, 17, 35, 53, 5, 30]  # C,N,O,S,P,F,Cl,Br,I,B,Zn

AA_LIST = [
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY",
    "HIS", "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER",
    "THR", "TRP", "TYR", "VAL",
]
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}

HYBRIDIZATION_LIST = ["SP", "SP2", "SP3", "SP3D", "SP3D2"]

RBF_CENTERS = np.linspace(0.0, 10.0, 21)
RBF_GAMMA = 5.0

NODE_DIM = 47  # 11 elem + 5 hybrid + 1 charge + 1 arom + 1 ring + 1 neighbors + 1 protein + 20 AA + 6 pad
EDGE_DIM = 24  # 21 RBF + 3 edge-type one-hot

# ─── Helpers ─────────────────────────────────────────────────────────────────

def _rbf_encode(distance: float) -> np.ndarray:
    return np.exp(-RBF_GAMMA * (distance - RBF_CENTERS) ** 2).astype(np.float32)


def _atom_features(
    atomic_num: int,
    hybridization: str | None = None,
    formal_charge: int = 0,
    is_aromatic: bool = False,
    is_in_ring: bool = False,
    num_heavy_neighbors: int = 0,
    is_protein: bool = False,
    residue_name: str | None = None,
) -> np.ndarray:
    feat = np.zeros(NODE_DIM, dtype=np.float32)
    idx = 0

    if atomic_num in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(atomic_num)] = 1.0
    idx += len(ELEMENT_LIST)

    if hybridization and hybridization in HYBRIDIZATION_LIST:
        feat[idx + HYBRIDIZATION_LIST.index(hybridization)] = 1.0
    idx += len(HYBRIDIZATION_LIST)

    feat[idx] = float(formal_charge);            idx += 1
    feat[idx] = 1.0 if is_aromatic else 0.0;     idx += 1
    feat[idx] = 1.0 if is_in_ring else 0.0;      idx += 1
    feat[idx] = float(num_heavy_neighbors);       idx += 1
    feat[idx] = 1.0 if is_protein else 0.0;      idx += 1

    if residue_name and residue_name in AA_TO_IDX:
        feat[idx + AA_TO_IDX[residue_name]] = 1.0

    return feat


def featurize_complex(pdb_text: str, ligand_sdf: str) -> Data | None:
    """Protein PDB + ligand SDF → PyG Data graph."""
    try:
        mol = Chem.MolFromMolBlock(ligand_sdf, removeHs=False)
        if mol is None:
            mol = Chem.MolFromMolBlock(ligand_sdf, removeHs=True)
        if mol is None:
            return None

        conf = mol.GetConformer()
        lig_coords, lig_features = [], []
        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            lig_coords.append([pos.x, pos.y, pos.z])
            hyb = str(atom.GetHybridization()) if atom.GetHybridization() else None
            lig_features.append(_atom_features(
                atomic_num=atom.GetAtomicNum(),
                hybridization=hyb,
                formal_charge=atom.GetFormalCharge(),
                is_aromatic=atom.GetIsAromatic(),
                is_in_ring=atom.IsInRing(),
                num_heavy_neighbors=len([n for n in atom.GetNeighbors() if n.GetAtomicNum() > 1]),
                is_protein=False,
            ))
        lig_coords = np.array(lig_coords)
        n_lig = len(lig_features)

        # Parse protein
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure("protein", io.StringIO(pdb_text))

        prot_atoms = [a for a in structure.get_atoms() if a.element.strip() not in ("H", "")]
        if not prot_atoms:
            return None

        prot_coords_all = np.array([a.get_vector().get_array() for a in prot_atoms])

        # Binding pocket: protein atoms within 10A of any ligand atom
        dist_matrix = cdist(prot_coords_all, lig_coords)
        pocket_mask = dist_matrix.min(axis=1) < 10.0
        pocket_indices = np.where(pocket_mask)[0]
        if len(pocket_indices) == 0:
            return None

        pocket_atoms = [prot_atoms[i] for i in pocket_indices]
        pocket_coords = prot_coords_all[pocket_indices]

        prot_features = []
        for atom in pocket_atoms:
            elem = atom.element.strip().capitalize()
            atomic_num = Chem.GetPeriodicTable().GetAtomicNumber(elem) if elem else 6
            residue = atom.get_parent()
            res_name = residue.get_resname() if residue else None
            prot_features.append(_atom_features(
                atomic_num=atomic_num,
                is_protein=True,
                residue_name=res_name,
            ))
        n_prot = len(prot_features)

        # Build graph
        all_features = np.array(lig_features + prot_features, dtype=np.float32)
        all_coords = np.vstack([lig_coords, pocket_coords])

        edge_src, edge_dst, edge_features = [], [], []

        # Intra-ligand bonds
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            dist = np.linalg.norm(all_coords[i] - all_coords[j])
            ef = np.concatenate([_rbf_encode(dist), [1, 0, 0]])
            edge_src.extend([i, j])
            edge_dst.extend([j, i])
            edge_features.extend([ef, ef])

        # Intra-protein bonds (< 1.6A)
        if n_prot > 1:
            prot_dist = cdist(pocket_coords, pocket_coords)
            pi, pj = np.where((prot_dist < 1.6) & (prot_dist > 0.1))
            for ii, jj in zip(pi, pj):
                ef = np.concatenate([_rbf_encode(prot_dist[ii, jj]), [0, 1, 0]])
                edge_src.append(n_lig + ii)
                edge_dst.append(n_lig + jj)
                edge_features.append(ef)

        # Protein-ligand interactions (< 5A)
        cross_dist = cdist(lig_coords, pocket_coords)
        li, pi = np.where(cross_dist < 5.0)
        for l_idx, p_idx in zip(li, pi):
            ef = np.concatenate([_rbf_encode(cross_dist[l_idx, p_idx]), [0, 0, 1]])
            edge_src.extend([l_idx, n_lig + p_idx])
            edge_dst.extend([n_lig + p_idx, l_idx])
            edge_features.extend([ef, ef])

        if not edge_src:
            return None

        return Data(
            x=torch.tensor(all_features, dtype=torch.float32),
            edge_index=torch.tensor([edge_src, edge_dst], dtype=torch.long),
            edge_attr=torch.tensor(np.array(edge_features, dtype=np.float32)),
            pos=torch.tensor(all_coords, dtype=torch.float32),
        )

    except Exception as e:
        print(f"featurize_complex failed: {e}")
        return None

print(f"NODE_DIM={NODE_DIM}, EDGE_DIM={EDGE_DIM}")

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

# Use different cache dir depending on pose source to avoid mixing
if USE_DIFFDOCK_POSES:
    CACHE_DIR = Path("/content/featurized_graphs_diffdock")
    print("Featurizing with DiffDock-predicted poses")
else:
    CACHE_DIR = Path("/content/featurized_graphs")
    print("Featurizing with crystal poses")

CACHE_DIR.mkdir(parents=True, exist_ok=True)

dataset = []
skipped = 0
skipped_no_dock = 0
pdb_codes = sorted(pkd_map.keys())

for pdb_code in tqdm(pdb_codes, desc="Featurizing"):
    cache_path = CACHE_DIR / f"{pdb_code}.pt"

    if cache_path.exists():
        cached = torch.load(cache_path, weights_only=False)
        dataset.append((cached["data"], cached["pkd"]))
        continue

    entry_dir = PDBBIND_DIR / pdb_code
    pdb_path = entry_dir / f"{pdb_code}_protein.pdb"

    # Choose ligand source: DiffDock pose or crystal pose
    if USE_DIFFDOCK_POSES:
        sdf_path = DOCKED_DIR / f"{pdb_code}_docked.sdf"
        if not sdf_path.exists():
            skipped_no_dock += 1
            continue
    else:
        sdf_path = entry_dir / f"{pdb_code}_ligand.sdf"

        # Some entries only have .mol2, convert via RDKit
        if not sdf_path.exists():
            mol2_path = entry_dir / f"{pdb_code}_ligand.mol2"
            if mol2_path.exists():
                try:
                    mol = Chem.MolFromMol2File(str(mol2_path), removeHs=False)
                    if mol is not None:
                        sdf_path.write_text(Chem.MolToMolBlock(mol))
                except Exception:
                    pass

    if not pdb_path.exists() or not sdf_path.exists():
        skipped += 1
        continue

    data = featurize_complex(pdb_path.read_text(), sdf_path.read_text())
    if data is None:
        skipped += 1
        continue

    pkd = pkd_map[pdb_code]
    data.y = torch.tensor([pkd], dtype=torch.float32)
    torch.save({"data": data, "pkd": pkd}, cache_path)
    dataset.append((data, pkd))

print(f"\nFeaturized: {len(dataset)}, skipped: {skipped}")
if USE_DIFFDOCK_POSES:
    print(f"Skipped (no DiffDock pose): {skipped_no_dock}")

if dataset:
    pkd_values = [pkd for _, pkd in dataset]
    print(f"pKd range: [{min(pkd_values):.2f}, {max(pkd_values):.2f}], "
          f"mean: {np.mean(pkd_values):.2f}, std: {np.std(pkd_values):.2f}")

## 3. Model

3-layer GIN with batch norm, concat(mean_pool, max_pool) readout, MLP head → scalar pKd.

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import GINConv, global_mean_pool, global_max_pool


class BindingAffinityGNN(nn.Module):
    def __init__(self, node_dim=47, edge_dim=24, hidden_dim=128):
        super().__init__()
        self.node_encoder = nn.Linear(node_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(3):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x = self.node_encoder(data.x)
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, data.edge_index)
            x = bn(x)
            x = x.relu()

        batch = data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x_cat = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=-1)
        return self.head(x_cat).squeeze(-1)


model = BindingAffinityGNN(node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=128)
print(f"BindingAffinityGNN: {sum(p.numel() for p in model.parameters()):,} parameters")

## 4. Training

80/10/10 split, Adam lr=1e-3, cosine annealing, early stopping (patience=20).

In [ ]:
import math, copy
from torch_geometric.loader import DataLoader

data_list = []
for data_obj, pkd in dataset:
    data_obj.y = torch.tensor([pkd], dtype=torch.float32)
    data_list.append(data_obj)

rng = np.random.RandomState(42)
indices = rng.permutation(len(data_list))

n_train = int(0.8 * len(data_list))
n_val = int(0.1 * len(data_list))

train_data = [data_list[i] for i in indices[:n_train]]
val_data   = [data_list[i] for i in indices[n_train:n_train + n_val]]
test_data  = [data_list[i] for i in indices[n_train + n_val:]]
print(f"Split: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")

BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_data, batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = BindingAffinityGNN(node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

NUM_EPOCHS = 200
PATIENCE = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_val_rmse = float("inf")
best_model_state = None
patience_counter = 0
train_rmse_history, val_rmse_history = [], []


def evaluate(loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds.append(model(batch).cpu())
            targets.append(batch.y.cpu().squeeze())
    preds = torch.cat(preds)
    targets = torch.cat(targets)
    return torch.sqrt(torch.mean((preds - targets) ** 2)).item()


for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    losses = []
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch), batch.y.squeeze())
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

    t_rmse = math.sqrt(np.mean(losses))
    v_rmse = evaluate(val_loader)
    train_rmse_history.append(t_rmse)
    val_rmse_history.append(v_rmse)

    if v_rmse < best_val_rmse:
        best_val_rmse = v_rmse
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        marker = " *"
    else:
        patience_counter += 1
        marker = ""

    if epoch % 5 == 0 or epoch == 1 or patience_counter == 0:
        lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Train: {t_rmse:.4f} | Val: {v_rmse:.4f} | LR: {lr:.6f}{marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Restored best model (val RMSE = {best_val_rmse:.4f})")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_rmse_history, label="Train RMSE")
ax.plot(val_rmse_history, label="Val RMSE")
ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE (pKd)")
ax.set_title("Training Curves")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Evaluation

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        all_preds.append(model(batch).cpu().numpy())
        all_targets.append(batch.y.cpu().squeeze().numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

test_rmse = np.sqrt(np.mean((all_preds - all_targets) ** 2))
test_mae = np.mean(np.abs(all_preds - all_targets))
pearson_r, pearson_p = pearsonr(all_targets, all_preds)

train_rmse = evaluate(train_loader)
val_rmse = evaluate(val_loader)

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Val RMSE:   {val_rmse:.4f}")
print(f"Test RMSE:  {test_rmse:.4f}")
print(f"Test MAE:   {test_mae:.4f}")
print(f"Pearson R:  {pearson_r:.4f} (p={pearson_p:.2e})")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(all_targets, all_preds, alpha=0.6, edgecolors="k", linewidths=0.3, s=40)
lims = [min(all_targets.min(), all_preds.min()) - 0.5, max(all_targets.max(), all_preds.max()) + 0.5]
ax.plot(lims, lims, "--", color="red", linewidth=1.5, label="y=x")
ax.set_xlabel("Actual pKd"); ax.set_ylabel("Predicted pKd")
ax.set_title(f"RMSE={test_rmse:.3f}, MAE={test_mae:.3f}, R={pearson_r:.3f}")
ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect("equal")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Export

Save weights + config for `backend/services/gnn_affinity.py`.

In [ ]:
export_path = "gnn_binding_affinity.pt"

torch.save({
    "state_dict": model.state_dict(),
    "hidden_dim": 128,
    "node_dim": NODE_DIM,
    "edge_dim": EDGE_DIM,
    "train_rmse": train_rmse,
    "val_rmse": val_rmse,
    "test_rmse": test_rmse,
    "pearson_r": pearson_r,
}, export_path)

file_size = os.path.getsize(export_path) / (1024 * 1024)
print(f"Saved: {export_path} ({file_size:.2f} MB)")

checkpoint = torch.load(export_path, map_location="cpu", weights_only=False)
print(f"  node_dim={checkpoint['node_dim']}, edge_dim={checkpoint['edge_dim']}, hidden_dim={checkpoint['hidden_dim']}")
print(f"  test_rmse={checkpoint['test_rmse']:.4f}, pearson_r={checkpoint['pearson_r']:.4f}")
print(f"\nCopy to backend/models/gnn_affinity/ for deployment.")